[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rafaellopesdesa/ML_Mandi/blob/main/activities/Lecture1_Classification_torch.ipynb)

## Setup

Run the setup cell once at the start of the session. A CPU runtime is sufficient. The data are generated here; no downloads, repository clone, or Google Drive mount are needed.

Run the cells in order and complete the marked exercises. The placeholders deliberately raise `NotImplementedError` until you fill them in.


In [ ]:
# Install required packages without upgrading existing installations.
#%pip -q install torch numpy matplotlib

import torch
print("PyTorch version:", torch.__version__)
print("Available device:", "cuda" if torch.cuda.is_available() else "cpu")


# Binary classification with PyTorch

Build and train a small neural network to distinguish two overlapping two-dimensional Gaussian distributions. This follows the regression activity, replacing the regression target with a binary label and the MSE loss with binary cross-entropy.

As in the PyTorch regression activity, we will build our own `nn.Module` from basic layers, write BCE from logits, and complete the training loop using automatic differentiation. We do not use a prebuilt classifier or pretrained model.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader

torch.manual_seed(22)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Training on:", device)


## A small, reproducible dataset

We want to distinguish two hypotheses with equal class probabilities:
$$H_0:\quad x\sim\mathcal N(\mu_0,I_2),\qquad
H_1:\quad x\sim\mathcal N(\mu_1,I_2),$$
$$\mu_0=(-0.75,-0.75),\qquad \mu_1=(0.75,0.75).$$

Here **unit Gaussian** means unit variance in each coordinate and zero covariance: the covariance matrix is the $2\times2$ identity. Labels are $y=0$ for $H_0$ and $y=1$ for $H_1$.

The supplied generator creates **2,000 events total**, with equal numbers of both classes in each partition: 1,200 training, 400 validation, and 400 test events. The partitions are independent draws. The same NumPy code and seed generate the same samples in all four notebooks; the two frameworks need not produce identical fitted weights.

Use training data to update weights, validation data to choose settings, and test data for the final evaluation. No external files are needed.


In [ ]:
# Shared verbatim by the JAX and PyTorch notebooks (students and solutions).
# Rerunning this cell recreates the same independent datasets.
DATA_SEED = 42
rng = np.random.default_rng(DATA_SEED)
mu0 = np.array([-0.75, -0.75])
mu1 = np.array([0.75, 0.75])

def make_dataset(n_per_class, rng):
    # Independent coordinates, each with standard deviation 1.
    x0 = rng.normal(loc=mu0, scale=1.0, size=(n_per_class, 2))
    x1 = rng.normal(loc=mu1, scale=1.0, size=(n_per_class, 2))
    x = np.concatenate([x0, x1]).astype(np.float32)
    y = np.concatenate([np.zeros(n_per_class), np.ones(n_per_class)]).astype(np.float32)
    order = rng.permutation(len(y))
    return x[order], y[order]

train_x, train_y = make_dataset(600, rng)
val_x, val_y = make_dataset(200, rng)
test_x, test_y = make_dataset(200, rng)
print("Training / validation / test events:", len(train_y), len(val_y), len(test_y))
print("Input shape:", train_x.shape, "label shape:", train_y.shape)


### Look at the two classes

The histograms and scatter plot show what the network must distinguish. Notice that the populations overlap; a classifier cannot perfectly separate new events from these distributions.


In [ ]:
# Inspect training data only; keep test data for the final evaluation.
fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))
colors = ["tab:blue", "tab:orange"]
bins = np.linspace(-4, 4, 33)
for label, color in enumerate(colors):
    points = train_x[train_y == label]
    axes[0].scatter(points[:, 0], points[:, 1], s=10, alpha=0.35,
                    color=color, label=f"Class {label}")
    for feature in range(2):
        axes[feature + 1].hist(points[:, feature], bins=bins, density=True,
                               histtype="step", color=color, label=f"Class {label}")
axes[0].set(xlabel="$x_1$", ylabel="$x_2$", title="Two overlapping Gaussians")
axes[0].set_aspect("equal", adjustable="box")
for feature in range(2):
    axes[feature + 1].set(xlabel=f"$x_{feature + 1}$", ylabel="Density")
for ax in axes:
    ax.legend()
fig.tight_layout()
plt.show()


## DataLoaders

Convert the inputs and labels to `float32` tensors. Mini-batches of 64 events keep the training loop similar to the regression notebook. Shuffle training examples; validation and test data do not need shuffling.


In [ ]:
train_ds = TensorDataset(torch.from_numpy(train_x), torch.from_numpy(train_y))
val_ds = TensorDataset(torch.from_numpy(val_x), torch.from_numpy(val_y))
test_ds = TensorDataset(torch.from_numpy(test_x), torch.from_numpy(test_y))

batch_size = 64
train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False)


## Build the neural network

Use two inputs, **one hidden layer with four `tanh` nodes**, and one linear output:
$$h=\tanh(xW_1+b_1),\qquad z=hW_2+b_2.$$

The scalar output $z$ is a **logit**, which can be any real number. Convert it to a class-1 probability with
$$f(x)=\operatorname{sigmoid}(z)=\frac{1}{1+e^{-z}}.$$

Keep the output linear when calculating the loss below; apply the sigmoid when you want probabilities. Both logits and labels will have shape `(N,)`.

**Exercise:** implement the network. How many trainable parameters does it have, including biases?


In [ ]:
class MLP(nn.Module):
    def __init__(self, input_dim=2, hidden_dim=4):
        super().__init__()
        # Exercise: define self.hidden and self.output with nn.Linear.
        raise NotImplementedError("Define the two linear layers.")

    def forward(self, x):
        # Exercise: apply hidden layer, torch.tanh, then output layer.
        # Use .squeeze(-1) to return logits of shape (N,), matching the labels.
        raise NotImplementedError("Implement the neural-network forward pass.")


In [ ]:
torch.manual_seed(22)
model = MLP().to(device)
print(model)
print("Trainable parameters:", sum(p.numel() for p in model.parameters()))
print("Logit shape for five examples:", model(train_ds.tensors[0][:5].to(device)).shape)


## Binary cross-entropy (BCE)

For labels $y_i\in\{0,1\}$ and predicted class-1 probabilities $f_i$, minimize
$$L=-\frac1N\sum_i\left[y_i\log f_i+(1-y_i)\log(1-f_i)\right].$$

A confident wrong prediction has a large loss. With $f_i=\operatorname{sigmoid}(z_i)$, the same loss is
$$L=\frac1N\sum_i\left[\log(1+e^{z_i})-y_i z_i\right].$$

**Exercise:** implement the mean BCE directly from logits with `logaddexp(0, logits) - targets * logits`. `logaddexp` computes the expression without overflowing for large positive logits or taking the logarithm of a probability rounded to zero. We use this form instead of clipping probabilities. Do not apply sigmoid before this loss.


In [ ]:
def bce_from_logits(logits, targets):
    # Exercise: use torch.logaddexp(torch.zeros_like(logits), logits)
    # and average over the batch with torch.mean.
    raise NotImplementedError("Implement the mean binary cross-entropy.")


In [ ]:
# A useful check: zero logits mean probability 0.5, so BCE should be log(2).
print("BCE at probability 0.5:", bce_from_logits(torch.zeros(4), torch.tensor([0., 1., 0., 1.])).item())
print("Expected:", np.log(2.0))


## Training with automatic differentiation

**Exercise:** complete the mini-batch training steps: reset gradients, calculate logits and loss, call `backward()`, and update parameters with the optimizer. These are the same operations as in the regression notebook.

The evaluation function is provided. It disables gradient tracking and never updates weights. Training loss averages batches while parameters change; validation loss uses the parameters at the end of each epoch.


In [ ]:
def train_one_epoch(model, loader, optimizer, loss_f):
    model.train()
    running_loss = 0.0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        # Exercise: zero_grad, forward pass, loss, backward, optimizer step.
        raise NotImplementedError("Complete the mini-batch training steps.")
        running_loss += loss.item() * len(xb)
    return running_loss / len(loader.dataset)

@torch.no_grad()
def evaluate(model, loader, loss_f):
    model.eval()
    running_loss = 0.0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        running_loss += loss_f(model(xb), yb).item() * len(xb)
    return running_loss / len(loader.dataset)


### Train the network

Use plain SGD with learning rate 0.1 for 100 epochs. Each epoch contains several mini-batch updates. **Exercise:** call the training and evaluation functions to fill the history arrays. Rerunning the cell reinitializes the model and optimizer.


In [ ]:
torch.manual_seed(22)
model = MLP().to(device)
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)
loss_f = bce_from_logits
history = {"loss": [], "val_loss": []}

for epoch in range(1, 101):
    # Exercise: call train_one_epoch and evaluate to obtain both losses.
    raise NotImplementedError("Run the training and validation passes.")
    history["loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    if epoch == 1 or epoch % 20 == 0:
        print(f"Epoch {epoch:03d} | train BCE={train_loss:.4f} | val BCE={val_loss:.4f}")


### Learning curves

Inspect the training and validation BCE. A lower validation BCE indicates better probability predictions on unseen events. Accuracy and BCE measure different aspects of performance.


In [ ]:
epochs = np.arange(1, len(history["loss"]) + 1)
plt.plot(epochs, history["loss"], label="training")
plt.plot(epochs, history["val_loss"], label="validation")
plt.axhline(np.log(2), color="gray", linestyle="--", label="constant probability 0.5")
plt.xlabel("Epoch")
plt.ylabel("Binary cross-entropy")
plt.legend()
plt.show()
print("Final training BCE:", history["loss"][-1])
print("Final validation BCE:", history["val_loss"][-1])


## Evaluate on independent test data

**Exercise:** obtain probabilities with sigmoid, predict class 1 when the probability is at least 0.5, and report test BCE and accuracy. Use the logits for BCE. Define `predict_probability(x)` to return a one-dimensional NumPy array so the common plots below work with either framework.


In [ ]:
@torch.no_grad()
def predict_probability(x):
    model.eval()
    x_tensor = torch.as_tensor(x, dtype=torch.float32, device=device)
    # Exercise: apply torch.sigmoid to the logits; return .cpu().numpy().
    raise NotImplementedError("Return class-1 probabilities.")

# Exercise: define test_prob, test_bce, and test_accuracy.
# evaluate(model, test_loader, loss_f) gives test BCE without training.
raise NotImplementedError("Evaluate the test set.")


## Compare with the optimal classifier

For a fixed $x$, the true label distribution is Bernoulli with probability $\eta(x)=P(y=1\mid x)$. Its expected BCE is the entropy of this Bernoulli distribution plus its KL divergence to the predicted Bernoulli distribution. The entropy does not depend on the network, so the population loss is minimized by $f^*(x)=\eta(x)$.

With equal class probabilities,
$$f^*(x)=\frac{p(x\mid H_1)}{p(x\mid H_0)+p(x\mid H_1)},\qquad
\frac{f^*(x)}{1-f^*(x)}=\frac{p(x\mid H_1)}{p(x\mid H_0)}.$$

For the two unit Gaussians, the log density ratio is
$$z^*(x)=\log\frac{p(x\mid H_1)}{p(x\mid H_0)}
=-\frac12\left[\lVert x-\mu_1\rVert^2-\lVert x-\mu_0\rVert^2\right].$$
Thus $f^*(x)=\operatorname{sigmoid}(z^*(x))$.

**Exercise:** implement `optimal_logit(x)` and `optimal_probability(x)`. For equal class priors and equal misclassification costs, the optimal boundary is $z^*=0$. Compare the network and analytic probabilities on the same test set. The optimum refers to the population expectation; a finite test sample can fluctuate.


In [ ]:
def optimal_logit(x):
    # Use the difference of squared distances to mu1 and mu0.
    raise NotImplementedError("Calculate the analytic log density ratio.")

def optimal_probability(x):
    # Apply the sigmoid to optimal_logit(x).
    raise NotImplementedError("Calculate the optimal class-1 probability.")


In [ ]:
optimal_test_logits = optimal_logit(test_x)
optimal_test_prob = optimal_probability(test_x)
optimal_test_bce = np.mean(np.logaddexp(0, optimal_test_logits)
                           - test_y * optimal_test_logits)
optimal_test_accuracy = np.mean((optimal_test_prob >= 0.5) == test_y)
print(f"Neural network: test BCE = {test_bce:.4f}, accuracy = {test_accuracy:.3f}")
print(f"Analytic optimum: test BCE = {optimal_test_bce:.4f}, accuracy = {optimal_test_accuracy:.3f}")

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
bins = np.linspace(0, 1, 31)
for label, color in enumerate(colors):
    axes[0].hist(test_prob[test_y == label], bins=bins, histtype="step",
                 color=color, label=f"Class {label}")
axes[0].set(xlabel="Predicted class-1 probability", ylabel="Events")
axes[0].legend()
axes[1].scatter(optimal_test_prob, test_prob, s=12, alpha=0.4)
axes[1].plot([0, 1], [0, 1], "k--")
axes[1].set(xlabel="Optimal probability", ylabel="NN probability",
            xlim=(0, 1), ylim=(0, 1))
fig.tight_layout()
plt.show()


### Visualize the decision boundary

The black contour marks probability 0.5. Compare both the boundary and the probability away from it. The scatter points are the held-out test events; the grid is used only for plotting.


In [ ]:
# A common plotting helper; no training takes place on this grid.
axis = np.linspace(-4, 4, 121)
xx, yy = np.meshgrid(axis, axis)
grid = np.column_stack([xx.ravel(), yy.ravel()]).astype(np.float32)
grid_prob = predict_probability(grid).reshape(xx.shape)
optimal_grid_prob = optimal_probability(grid).reshape(xx.shape)

fig, axes = plt.subplots(1, 2, figsize=(10, 4), layout="constrained")
for ax, prob, title in zip(axes, [grid_prob, optimal_grid_prob],
                          ["Trained neural network", "Analytic optimum"]):
    surface = ax.contourf(xx, yy, prob, levels=np.linspace(0, 1, 21),
                          cmap="RdBu_r", vmin=0, vmax=1)
    ax.contour(xx, yy, prob, levels=[0.5], colors="black")
    for label, color in enumerate(colors):
        points = test_x[test_y == label]
        ax.scatter(points[:, 0], points[:, 1], s=8, color=color,
                    edgecolors="white", linewidths=0.2, alpha=0.6)
    ax.set(xlabel="$x_1$", ylabel="$x_2$", title=title, xlim=(-4, 4), ylim=(-4, 4))
    ax.set_aspect("equal")
fig.colorbar(surface, ax=axes, label="Class-1 probability")
plt.show()


## Questions and experiments

1. Count the trainable parameters. Which belong to the weights and which to the biases?
2. Why should neither the optimal BCE nor the classification error be zero for these overlapping distributions?
3. Expand the squared distances in $z^*$. What is the decision boundary? Is a hidden layer necessary for this particular problem?
4. What happens if the two means coincide? What happens when you move them farther apart?
5. Try changing the learning rate, number of epochs, or hidden-layer width. Use validation loss to compare settings. What would a rising validation loss with a falling training loss suggest?
6. If the training class proportions were unequal, would the classifier odds still equal the density ratio?

The supplied implementation uses equal class probabilities and a 0.5 decision threshold. Keep the test set out of hyperparameter choices. For a new experiment with different Gaussian means, regenerate all three partitions using the fixed seed.
